In [1]:
import collections
import numpy as np
import pandas as pd
import re

from argparse import Namespace

#### Added to make sure cwd is at book root folder:

In [4]:
# ADDED
from pathlib import Path
import os
cwd = Path.cwd()
while cwd.name != "PyTorchNLPBook":
    if cwd.parent == cwd: raise RutimeError("Couldn't find PyTorchNLPBook")
    cwd = cwd.parent
os.chdir(cwd)
Path.cwd()

WindowsPath('D:/Source/gh-pc/PyTorchNLPBook')

In [5]:
args = Namespace(
    raw_train_dataset_csv="data/yelp/raw_train.csv",
    raw_test_dataset_csv="data/yelp/raw_test.csv",
    proportion_subset_of_train=0.1,
    train_proportion=0.7,
    val_proportion=0.15,
    test_proportion=0.15,
    output_munged_csv="data/yelp/reviews_with_splits_lite.csv",
    seed=1337
)

In [ ]:
# Read raw data
train_reviews = pd.read_csv(args.raw_train_dataset_csv, header=None, names=['rating', 'review'])

### Overview of data (added)
`train_reviews` is a pandas.Dataframe with shape (560000,2). Column "rating" has type `int64`; column "review" has type `str`. Rating values: [1, 2]; there are 280000 rows for each. 

In [ ]:
# ADDED
print(f"train_reviews.shape: {train_reviews.shape}")
train_reviews.info()


In [ ]:
# ADDED 
train_reviews.iloc[0]

In [ ]:
# ADDED
type(train_reviews.iloc[0])

In [ ]:
# ADDED
train_reviews.iloc[0].to_dict()

In [ ]:
# making the subset equal across the review classes
by_rating = collections.defaultdict(list)
for _, row in train_reviews.iterrows():
    by_rating[row.rating].append(row.to_dict())
    
review_subset = []

for _, item_list in sorted(by_rating.items()):

    n_total = len(item_list)
    n_subset = int(args.proportion_subset_of_train * n_total)
    review_subset.extend(item_list[:n_subset])

review_subset = pd.DataFrame(review_subset)

In [ ]:
# ADDED
by_rating.keys()

In [ ]:
# ADDED
print(type(by_rating[1]))
len(by_rating[1])

In [ ]:
# ADDED
by_rating[1][0]

In [ ]:
# ADDED
len(review_subset)

In [ ]:
# ADDED
n_subset

In [ ]:
# ADDED
print(f"review_subset.shape: {review_subset.shape}")
review_subset.info()

In [ ]:
review_subset.head()

In [ ]:
train_reviews.rating.value_counts()

In [ ]:
review_subset.rating.value_counts()

In [ ]:
# Unique classes
set(review_subset.rating)

#### Comment on following (added)
As above `by_rating` will be a dict with two entries (keys {1, 2}), and each entry will be a list of dicts corresponding to the DF rows (Series). 

The `for` loop with iterate over the two entries. For each, the list (DF row items) will be randomly shuffled, and then split into train, val and test sets.

In [ ]:
# Splitting the subset by rating to create our new train, val, and test splits
by_rating = collections.defaultdict(list)
for _, row in review_subset.iterrows():
    by_rating[row.rating].append(row.to_dict())
    
final_list = []
np.random.seed(args.seed)

for _, item_list in sorted(by_rating.items()):

    np.random.shuffle(item_list)
    
    n_total = len(item_list)
    n_train = int(args.train_proportion * n_total)
    n_val = int(args.val_proportion * n_total)
    n_test = int(args.test_proportion * n_total)
    
    # Give data point a split attribute
    for item in item_list[:n_train]:
        item['split'] = 'train'
    
    for item in item_list[n_train:n_train+n_val]:
        item['split'] = 'val'
        
    for item in item_list[n_train+n_val:n_train+n_val+n_test]:
        item['split'] = 'test'

    # Add to final list
    final_list.extend(item_list)

#### Duplication
The following block duplicates most of the preceding code block

In [ ]:
# Create split data
final_list = []
np.random.seed(args.seed)

for _, item_list in sorted(by_rating.items()):

    np.random.shuffle(item_list)
    
    n_total = len(item_list)
    n_train = int(args.train_proportion * n_total)
    n_val = int(args.val_proportion * n_total)
    n_test = int(args.test_proportion * n_total)
    
    # Give data point a split attribute
    for item in item_list[:n_train]:
        item['split'] = 'train'
    
    for item in item_list[n_train:n_train+n_val]:
        item['split'] = 'val'
        
    for item in item_list[n_train+n_val:n_train+n_val+n_test]:
        item['split'] = 'test'

    # Add to final list
    final_list.extend(item_list)

In [ ]:
# ADDED
print(len(final_list))
print(final_list[0])
final_list[-1]

In [ ]:
# Write split data to file
final_reviews = pd.DataFrame(final_list)

In [ ]:
# ADDED
print(f"final_reviews.shape: {final_reviews.shape}")
final_reviews.info()

In [ ]:
final_reviews.split.value_counts()

#### Comment on following (added)
Substitutions add space before basic punctuation (`[.,!?]`), then replace any "special" characters with space.

`final_reviews.review` returns the "review" column from the pandas.Dataframe (the columns is a Series object), and `.apply()` applies the specified function to each element in the Series. Then the `review` column is updated with the new data.

In [ ]:
# Preprocess the reviews
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"([.,!?])", r" \1 ", text)
    text = re.sub(r"[^a-zA-Z.,!?]+", r" ", text)
    return text
    
final_reviews.review = final_reviews.review.apply(preprocess_text)

In [ ]:
# ADDED
print(final_reviews.review.shape)
type(final_reviews.review)

In [ ]:
# ADDED
final_reviews.review.info()

#### Comment on the following (added)
Like the above, the elements in the `rating` column are modified with replacement values. 

`{...}.get` behaves like a reference to a `mapping.get()` method; the current value is passed as a parameter, and the mapped value is returned.

Note two means of referring to the dataframe column: `final_reviews['rating']` and 'final_reviews.rating`. The latter is a convenience provided by Pandas, but only if the name is valid as a Python identifier and don't conflice with any existing method or attribute. Referencing using the name key is generally recommended.

In [ ]:
final_reviews['rating'] = final_reviews.rating.apply({1: 'negative', 2: 'positive'}.get)

In [ ]:
# ADDED
final_reviews['rating'].value_counts()

In [ ]:
# ADDED
final_reviews['split'].value_counts()

In [ ]:
final_reviews.head()

#### Comment on the following (added)
This will write out the dataframe to a csv file, "data/yelp/reviews_with_splits_lite.csv". This file already existed in the data; no new file is created. Also, because the same random seed is used, the results will be the same, so nothing will have changed (except the file timestamp).

In [ ]:
final_reviews.to_csv(args.output_munged_csv, index=False)